# HGRIA - Hand Gesture Recognition for Interactive Applications
## Google Colab Launch Notebook

```
┌─────────────────────────────────────────────────────────────┐
│                    ARCHITECTURE (COLAB)                     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   Browser (Frontend)     ngrok Tunnel      Colab Backend   │
│   ┌──────────────┐      ┌──────────┐     ┌──────────────┐   │
│   │  GitHub Pages │ ←─── │  HTTPS   │ ←── │  Flask +     │   │
│   │  / Vercel    │      │  Tunnel  │     │  MediaPipe   │   │
│   └──────────────┘      └──────────┘     └──────────────┘   │
│         │                                          │        │
│         │           Google Drive                    │        │
│         └──────────────┬───────────────────────────┘        │
│                        │ logs/                             │
└─────────────────────────────────────────────────────────────┘
```

### Prerequisites
- Google Account with Google Drive access
- HGRIA project saved in Google Drive at `/MyDrive/HGRIA/`
- ngrok account (free tier works)
- WebRTC-compatible browser (Chrome, Edge, Firefox)

### How it works
1. Project is copied from Google Drive to Colab runtime
2. Backend runs Flask server on Colab with MediaPipe
3. ngrok creates HTTPS tunnel to expose backend
4. Frontend connects via WebSocket and sends webcam frames
5. Backend processes frames and sends gesture commands back

In [ ]:
# Step 1: Install pinned dependencies
# Run this first — Colab may have mismatched versions pre-installed
!pip install --no-cache-dir \
    "numpy==1.26.4" \
    "tensorflow==2.18.0" \
    "protobuf==4.25.3" \
    "mediapipe==0.10.21" \
    "opencv-contrib-python==4.11.0.86"

In [ ]:
# Step 2: Verify installed versions
import numpy as np
import google.protobuf
import mediapipe as mp
import tensorflow as tf

print("NumPy:", np.__version__)
print("Protobuf:", google.protobuf.__version__)
print("MediaPipe:", mp.__version__)
print("TensorFlow:", tf.__version__)
print("MediaPipe OK")
print("TensorFlow OK")

In [ ]:
# Step 3: Mount Google Drive and verify project files
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

SOURCE_PATH = '/content/drive/MyDrive/HGRIA'
REQUIREMENTS_FILE = os.path.join(SOURCE_PATH, 'requirements.txt')

if not os.path.exists(REQUIREMENTS_FILE):
    raise FileNotFoundError(
        f"requirements.txt not found at {REQUIREMENTS_FILE}.\n"
        "Please ensure HGRIA project is saved in your Google Drive at:\n"
        "  /content/drive/MyDrive/HGRIA/"
    )

print(f"✓ Project files found at {SOURCE_PATH}")

In [ ]:
# Step 4: Copy project to Colab runtime and install dependencies
import shutil
import subprocess
import sys

DEST = '/content/HGRIA'

if os.path.exists(DEST):
    print(f"Project already exists at {DEST}")
else:
    shutil.copytree(SOURCE_PATH, DEST)
    print(f"✓ Copied project to {DEST}")

sys.path.insert(0, DEST)
os.chdir(DEST)

result = subprocess.run(
    ['pip', 'install', '-q', '-r', 'requirements.txt'],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        f"Failed to install dependencies:\n{result.stderr}"
    )

print("✓ Dependencies installed successfully")

In [ ]:
# Step 5: Configure ngrok authentication
import getpass

subprocess.run(['pip', 'install', '-q', 'pyngrok'], capture_output=True)
from pyngrok import ngrok

print("Enter your ngrok authtoken (from https://dashboard.ngrok.com/auth)")
print("Press Enter to skip (anonymous tunnel may disconnect)")

authtoken = getpass.getpass(prompt='Authtoken: ')

if authtoken:
    ngrok.set_auth_token(authtoken)
    print("✓ ngrok authenticated")
else:
    print("⚠ Anonymous tunnel - connection may be unstable")

In [ ]:
# Step 6: Start ngrok tunnel and display connection info
tunnel = ngrok.connect(5000, "http")
ngrok_url = tunnel.public_url.replace('http://', 'https://')

print("=" * 60)
print("🔗 NGROK TUNNEL READY")
print("=" * 60)
print(f"\nBackend URL: {ngrok_url}")

frontend_script = f'<script>window.HGRIA_BACKEND_URL="{ngrok_url}";</script>'
print(f"\nPaste this in your frontend HTML (before Socket.IO loads):")
print(f"\n{frontend_script}")

frontend_base = "https://qtannguyen-researcher.github.io/HGRIA"
frontend_url = f"{frontend_base}?server={ngrok_url}"

print(f"\nOr open Frontend directly with:")
print(f"\n{frontend_url}")
print("\n" + "=" * 60)

In [ ]:
# Step 7: Patch config for Colab environment
import json
from pathlib import Path

PROJECT_ROOT = Path(DEST)
config_path = str(PROJECT_ROOT / 'config' / 'config.json')

with open(config_path, 'r') as f:
    config = json.load(f)

config['camera']['colab_mode'] = True
config['server']['cors_origins'] = '*'
config['logging']['log_to_file'] = True
config['logging']['log_file_path'] = '/content/drive/MyDrive/HGRIA/logs/'

# Ensure logs directory exists on Drive
os.makedirs(config['logging']['log_file_path'], exist_ok=True)

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("✓ Config patched:")
print(f"  - colab_mode   : {config['camera']['colab_mode']}")
print(f"  - cors_origins : {config['server']['cors_origins']}")
print(f"  - log_file_path: {config['logging']['log_file_path']}")

In [ ]:
# Step 8: Start the HGRIA Server (blocking)
# This cell will keep running until interrupted
from backend.main import SystemOrchestrator

print("Starting HGRIA Backend Server...")
print(f"Server running at: {ngrok_url}")
print("\nPress Stop button (■) to terminate the server")
print("-" * 40)

orchestrator = SystemOrchestrator(config_path)
orchestrator.start()

## Post-Launch Instructions

### Accessing the Frontend

After the server starts, open your browser and navigate to:

```
https://qtannguyen-researcher.github.io/HGRIA/?server=<NGROK_URL>
```

### If ngrok URL Changes

1. Stop the server (interrupt cell 8)
2. Re-run cells 6, 7, and 8 in sequence
3. Update the frontend with the new URL

### Troubleshooting

| Issue | Solution |
|-------|----------|
| Colab session timeout | Re-run cell 8 (server restart is fast) |
| ngrok URL changed | Re-run cells 6, 7, 8 and update frontend |
| Webcam denied | Use keyboard fallback (Arrow keys, Space, P, S) |
| High latency | Check Colab GPU availability (Runtime > Change runtime type) |
| Drive not mounted | Re-run cell 3 |

### Keyboard Controls (Fallback)
- Arrow Keys: Move
- Space: Jump
- P: Pause
- S: Speed Boost
- Enter: Confirm